In [1]:
import numpy as np
from scipy.stats import gaussian_kde

import jax
import jax.numpy as jnp
import jax.scipy as jsp
import jax.random as jr
jax.config.update("jax_enable_x64", True)
import equinox as eqx
import optax
import matplotlib.pyplot as plt
from tqdm import trange

from flowjax.distributions import StandardNormal, Transformed
from flowjax.bijections import Coupling, RationalQuadraticSpline, Chain, Permute, Affine, TriangularAffine

In [2]:
# ==========================================
# 1. Definição do Dataset GMM 3D
# ==========================================
class Params:
    def __init__(self):
        self.asymptotic = 200.0
        self.inflexion = 500.0
        self.tau = 150.0
        self.cov_latent = jnp.diag(jnp.array([40.0, 100.0, 100.0]))

def sample_gmm_data(key, n):
    p = Params()
    key1, key_mean, key_comp, key_norm = jax.random.split(key, 4)
    
    base_mean = jnp.array([p.asymptotic, p.inflexion, p.tau])
    L = jnp.linalg.cholesky(p.cov_latent)
    spread = 100.0 
    
    gmm_means = base_mean + jax.random.normal(key_mean, shape=(3, 3)) * spread
    components = jax.random.randint(key_comp, shape=(n, 3), minval=0, maxval=3)
    assigned_means = gmm_means[components, jnp.arange(3)] 
    
    z = jax.random.normal(key_norm, shape=(n, 3)) @ L.T + assigned_means
    return z

key = jr.PRNGKey(42)
key, subkey = jr.split(key)
num_samples = 10000
x_train = sample_gmm_data(subkey, num_samples)

print(f"Generated {num_samples} 3D samples.")
print(f"Data Means: {jnp.mean(x_train, axis=0)}")
print(f"Data Std: {jnp.std(x_train, axis=0)}\n")

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


Generated 10000 3D samples.
Data Means: [273.31749332 607.15079226 150.41357565]
Data Std: [ 41.04529785 111.14218912 109.17680424]



In [ ]:
# ==========================================
# 2. Construção da Arquitetura Multi-Layer com Dropout
# ==========================================
bound = 5.0
K = 8
num_layers = 8  # Número de coupling layers
base_dist = StandardNormal((3,))

loc = jnp.mean(x_train, axis=0)
scale = jnp.std(x_train, axis=0)
renormalizer = Affine(loc=loc, scale=scale)

# Custom Equinox Conditioner with Internal State
class DropoutConditioner(eqx.Module):
    layers: list
    dropout: eqx.nn.Dropout
    key: jax.Array
    inference: bool

    def __init__(self, in_dim, out_dim, hidden_dim, dropout_rate, key):
        k1, k2 = jr.split(key)
        self.layers = [
            eqx.nn.Linear(in_dim, hidden_dim, key=k1),
            eqx.nn.Linear(hidden_dim, out_dim, key=k2)
        ]
        self.dropout = eqx.nn.Dropout(p=dropout_rate)
        self.key = jr.PRNGKey(0)  # Dummy key placeholder
        self.inference = True     # Default to eval mode

    def __call__(self, x):
        # Uses internal state injected via eqx.tree_at during training
        x = jax.nn.relu(self.layers[0](x))
        x = self.dropout(x, key=self.key, inference=self.inference)
        x = self.layers[1](x)
        return x

layers = []
flow_key = jr.PRNGKey(127)

for i in range(num_layers):
    flow_key, subkey1, subkey2, cond_key = jr.split(flow_key, 4)
    
    transformer = RationalQuadraticSpline(knots=K, interval=bound)
    
    coupling = Coupling(
        key=subkey1,
        transformer=transformer,
        untransformed_dim=2,
        dim=3,
        nn_width=32, 
        nn_depth=2
    )
    
    out_dim = coupling.conditioner.layers[-1].weight.shape[0]

    
    conditioner = DropoutConditioner(
        in_dim=2, 
        out_dim=out_dim, 
        hidden_dim=32, 
        dropout_rate=0.2, 
        key=cond_key
    )
    
    # 1. Initialize Coupling with the required dummy network arguments

    out_dim = coupling.conditioner.layers[-1].weight.shape[0]
    
    # 2. Swap out the default MLP for our DropoutConditioner
    coupling = eqx.tree_at(
        lambda c: c.conditioner, 
        coupling, 
        conditioner
    )
    
    layers.append(coupling)
    layers.append(TriangularAffine(jnp.zeros(3), jnp.eye(3)))
    
    permutation = jax.random.permutation(subkey2, jnp.arange(3))
    layers.append(Permute(permutation))

layers.append(renormalizer)

bijection = Chain(layers)
flow = Transformed(base_dist, bijection)

AttributeError: 'RationalQuadraticSpline' object has no attribute 'num_params'

In [ ]:
# 1. Train/Validation Split (80/20)
num_train = int(0.8 * num_samples)
x_train_split = x_train[:num_train]
x_val_split = x_train[num_train:]

# 2. Add Regularization via AdamW (Weight Decay)
learning_rate = 5e-3
weight_decay = 1e-4
optim = optax.adamw(learning_rate, weight_decay=weight_decay)
opt_state = optim.init(eqx.filter(flow, eqx.is_inexact_array))

# 3. Dynamic State Injection Helper
def set_dropout_state(model, key=None, inference=True):
    is_dropout = lambda n: isinstance(n, DropoutConditioner)
    
    def update_node(node):
        node = eqx.tree_at(lambda c: c.inference, node, inference)
        if key is not None:
            node = eqx.tree_at(lambda c: c.key, node, key)
        return node

    return jax.tree_util.tree_map(
        lambda n: update_node(n) if is_dropout(n) else n,
        model,
        is_leaf=is_dropout
    )

# 4. Component Extraction Function
@eqx.filter_vmap(in_axes=(None, 0, 0))
def compute_components_train(model, x, key):
    # Inject the key and set to training mode
    model_train = set_dropout_state(model, key=key, inference=False)
    z, log_det = model_train.bijection.inverse_and_log_det(x)
    base_log_prob = model_train.base_dist.log_prob(z)
    return base_log_prob, log_det

@eqx.filter_vmap(in_axes=(None, 0))
def compute_components_eval(model, x):
    # Ensure dropout is disabled for evaluation
    model_eval = set_dropout_state(model, inference=True)
    z, log_det = model_eval.bijection.inverse_and_log_det(x)
    base_log_prob = model_eval.base_dist.log_prob(z)
    return base_log_prob, log_det

@eqx.filter_value_and_grad(has_aux=True)
def loss_fn(model, x, dropout_key):
    # Generate a unique key for every item in the batch
    batch_size = x.shape[0]
    keys = jax.random.split(dropout_key, batch_size)
    
    base_log_prob, log_det = compute_components_train(model, x, keys)
    avg_base = jnp.mean(base_log_prob)
    avg_det = jnp.mean(log_det)
    loss = -(avg_base + avg_det)
    return loss, (avg_base, avg_det)

@eqx.filter_jit
def train_step(model, state, x, dropout_key):
    params = eqx.filter(model, eqx.is_inexact_array)
    (loss, (avg_base, avg_det)), grads = loss_fn(model, x, dropout_key)
    updates, new_state = optim.update(grads, state, params)
    model = eqx.apply_updates(model, updates)
    return model, new_state, loss, avg_base, avg_det

@eqx.filter_jit
def eval_step(model, x):
    base_log_prob, log_det = compute_components_eval(model, x)
    return -jnp.mean(base_log_prob + log_det)

# 5. Training Loop with Tracking
epochs = 3000
batch_size = 1024
num_batches = num_train // batch_size

history = {
    "train_loss": [], "val_loss": [], 
    "base_log_prob": [], "log_det": []
}

pbar = trange(epochs, desc="Training Flow")
for epoch in pbar:
    key, subkey, drop_key = jax.random.split(key, 3)
    indices = jax.random.permutation(subkey, num_train)
    
    epoch_train_loss, epoch_base, epoch_det = 0.0, 0.0, 0.0
    
    for i in range(num_batches):
        batch_indices = indices[i * batch_size : (i + 1) * batch_size]
        x_batch = x_train_split[batch_indices]
        
        # Split a new dropout key for each batch block
        drop_key, batch_drop_key = jax.random.split(drop_key)
        
        flow, opt_state, loss, avg_base, avg_det = train_step(flow, opt_state, x_batch, batch_drop_key)
        epoch_train_loss += loss
        epoch_base += avg_base
        epoch_det += avg_det
        
    avg_train_loss = epoch_train_loss / num_batches
    avg_val_loss = eval_step(flow, x_val_split)
    
    history["train_loss"].append(avg_train_loss)
    history["val_loss"].append(avg_val_loss)
    history["base_log_prob"].append(epoch_base / num_batches)
    history["log_det"].append(epoch_det / num_batches)
    
    pbar.set_postfix(t_loss=f"{avg_train_loss:.2f}", v_loss=f"{avg_val_loss:.2f}")

In [ ]:
# ==========================================
# 5. Unconditional Sampling and KDE Visualization
# ==========================================
# Draw 10,000 samples from the trained normalizing flow
sample_key = jax.random.key(999)
num_generated_samples = 100000
x_generated = flow.sample(sample_key, (num_generated_samples,))

dim_names = [
    "Dimension 0 (Asymptotic ~ 200)", 
    "Dimension 1 (Inflexion ~ 500)", 
    "Dimension 2 (Tau ~ 150)"
]

for d in range(3):
    plt.figure(figsize=(10, 6))
    
    # Convert to NumPy for SciPy KDE
    train_data_1d = np.array(x_train[:, d])
    gen_data_1d = np.array(x_generated[:, d])
    
    # Create an evaluation grid based on data bounds
    min_val = min(train_data_1d.min(), gen_data_1d.min())
    max_val = max(train_data_1d.max(), gen_data_1d.max())
    x_grid = np.linspace(min_val, max_val, 1000)
    
    # Fit KDEs
    kde_train = gaussian_kde(train_data_1d)
    kde_gen = gaussian_kde(gen_data_1d)
    
    # Plot KDE curves
    plt.plot(x_grid, kde_train(x_grid), color='gray', linewidth=3, linestyle='--', label="Training Data KDE")
    plt.plot(x_grid, kde_gen(x_grid), color='blue', linewidth=2.5, alpha=0.8, label="Generated Flow KDE")
    
    # Add light histograms in the background for visual context
    plt.hist(train_data_1d, bins=100, density=True, alpha=0.15, color='gray')
    plt.hist(gen_data_1d, bins=100, density=True, alpha=0.15, color='blue')
    
    plt.title(f"3D Spline Flow Unconditional Sampling: {dim_names[d]}")
    plt.xlabel("Original Space")
    plt.ylabel("Density")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()